# Intrinsic Camera / Telescope Split — all Zernikes (v2)

**Author:** Aaron Roodman
**Date Created:** 2026-06-10
**Last Modified:** 2026-06-10
**Status:** Draft
**Keywords:** AOS, FAM, intrinsic, Zernike, rotator, OCS, CCS, decomposition, spin

## Description

Decompose the measured intrinsic Zernike maps into a **telescope-fixed (OCS)**
component `O` and a **camera-fixed (CCS)** component `C` that rotates with the
rotator, using FAM intrinsic maps built at several rotator angles.  Runs for
every Zernike in `noll_list` (default Noll 4–19, 22–26; 20/21 excluded).

Each Zernike of azimuthal order `n` (the *spin*) is handled correctly: scalar
terms (Z4, Z11, Z22; n=0) decompose as a real field, while doublets
(astig n=2, coma n=1, trefoil n=3, …) are combined into the complex field
`A = Z_cos + i·Z_sin` and decomposed with the spin model

```
A_θ(r,φ) = O(r,φ) + e^{i·n·sθ} · C(r, φ − sθ)   ⇒   A_{d,m} = O_m + C_m e^{i(n−m)sθ_d}
```

so the camera component's Zernike coefficients are allowed to rotate as the
camera turns.  Hole-aware least-squares (`code/intrinsic_split.decompose_spin_lsq`)
keeps `O` hole-free.

**Output:** per-Zernike page-1 maps (data / O / C / residual, RMS-annotated), a
telescope-vs-camera RMS summary across all Zernikes, and a parquet of every
`O`/`C` map.  Files in `output/intrinsic_split/`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-10 | Aaron Roodman | Initial version: Fourier-per-radius decomposition of the Z4-optical intrinsic map into telescope-fixed (OCS) and camera-fixed (CCS) components from FAM maps at 5 rotator angles; auto rotation-sign, m0_assignment flag, actual mean rotator from the fits parquet, diagnostics + maps + output parquet. |
| 2026-06-10 (v2) | Aaron Roodman | Added hole-aware decomposition (decompose_polar_lsq): per-radius masked Fourier least-squares fits O and C from valid samples only, so the telescope map O is hole-free while the camera-dead CCS regions are smoothly filled in C.  Now the default (decomp_method='lsq', hole_dist=0.06 deg).  Verified sign convention OCS<-CCS = +1*rotator from the donut parquet (matches auto s=+1). |
| 2026-06-10 (v3) | Aaron Roodman | Bumped reconstruction r_max to 1.75 deg with denser radial sampling (n_r=80) so O fills to ~1.73 deg (last well-covered radius; the ~empty 1.75 ring is skipped) and added light ridge regularization (ridge=1e-3) so partially-covered outer rings do not ring. Added a detailed markdown explanation of the Fourier-per-radius fit and a diagnostics page (camera azimuthal-mode spectrum, axisymmetric m=0 radial profile, residual-vs-radius). |
| 2026-06-10 (v4) | Aaron Roodman | Generalised to all Zernikes (Noll 4–19,22–26): added the spin-aware complex decomposition (`decompose_spin_fft` / hole-aware `decompose_spin_lsq`) so astig/coma/trefoil/… doublets are combined as `Z_cos+i·Z_sin` and the camera component is allowed to rotate as spin |m| (degenerate mode shifts m=0→m=n). Per-Zernike page-1 plots now annotate the map RMS of data / model / residual / |O| / |C|, plus a telescope-vs-camera RMS summary across all Zernikes. |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Decomposition (all Zernikes)](#decomp)
6. [Maps & RMS summary](#plots)
7. [Output](#output)

<a id='params'></a>
## 1. Parameters

In [1]:
# ============================================================
# Parameters
# ============================================================
intrinsic_base = 'output/build_measured_intrinsic/fam_danish_1_0_wep17_3_0_bin2x'
grid_filename  = 'measured_intrinsic_nkeep_34_pathA_grid.parquet'
datasets = [
    {'label': 'rot-60', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_-65.0_-55.0',
     'rot_range': (-65.0, -55.0), 'alt_range': (55.0, 75.0)},
    {'label': 'rot-15', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_-20.0_-10.0',
     'rot_range': (-20.0, -10.0), 'alt_range': (55.0, 75.0)},
    {'label': 'rot00',  'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_-3.0_3.0',
     'rot_range': (-3.0,  3.0),  'alt_range': (55.0, 75.0)},
    {'label': 'rot+15', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_10.0_20.0',
     'rot_range': (10.0, 20.0),  'alt_range': (55.0, 75.0)},
    {'label': 'rot+60', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_55.0_65.0',
     'rot_range': (55.0, 65.0),  'alt_range': (55.0, 75.0)},
]

# Zernikes to analyse (Noll).  20 and 21 (pentafoil) excluded by default.
noll_list = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
             22, 23, 24, 25, 26]
# Use the CCD-height-removed Z4 (z4_optical_OCS) for Z4; raw zk for the rest
# (CCD height feeds mostly Z4).
use_z4_optical = True

# FAM fits parquet(s) for the actual mean rotator per dataset.
fits_parquet_paths = [
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260315_20260327_fits.parquet',
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260331_20260409_fits.parquet',
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260418_20260531_fits.parquet',
]
rotator_angle_source = 'data'    # 'data' | 'dirname_center'

# Degenerate m=n mode -> telescope by default.
degen_assignment = 'ocs'         # 'ocs' | 'ccs' | 'split'
rotation_sign = 'auto'           # field-rotation sense s; 'auto' from Z4, or +-1
# Camera doublet spin sense: physical is +|m| (auto-confirmed +1 on astig/coma).
spin_sign = 1                    # +1 | -1
m_max = 12                       # azimuthal-order cap (1+4*m_max real unknowns/ring)
ridge = 1e-3                     # Tikhonov damping for partially-covered rings
hole_dist = 0.06                 # deg; nodes farther than this from a real bin are holes

polar = dict(r_min=0.06, r_max=1.75, n_r=80, n_az=180)
r_lim = (0.1, 1.6)               # radial window for RMS / residual metrics

output_dir     = 'output/intrinsic_split'
output_pdf     = f'{output_dir}/intrinsic_split_allz.pdf'
output_parquet = f'{output_dir}/intrinsic_split_allz_maps.parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, 'code')
import intrinsic_split as isp
from common.utils import setup_plotting

setup_plotting()
print('Ready.')

Ready.


<a id='functions'></a>
## 3. Helper Functions

In [3]:
def plot_field(ax, X, Y, vals, title, vlim, levels=21, cmap='RdBu_r'):
    """Filled-contour map of a field on the polar-grid points (X, Y)."""
    xr, yr, vr = X.ravel(), Y.ravel(), np.asarray(vals).ravel()
    fin = np.isfinite(vr)                       # drop NaN (skipped/hole nodes)
    im = ax.tricontourf(xr[fin], yr[fin], vr[fin],
                        levels=np.linspace(-vlim, vlim, levels),
                        cmap=cmap, extend='both')
    ax.set_aspect('equal'); ax.set_title(title, fontsize=9)
    ax.set_xlabel('thx [deg]', fontsize=7); ax.set_ylabel('thy [deg]', fontsize=7)
    ax.tick_params(labelsize=6)
    return im


def map_rms(v, R, r_lim):
    """RMS of |v| over polar rings within r_lim (NaN-aware)."""
    m = (R >= r_lim[0]) & (R <= r_lim[1])
    return float(np.sqrt(np.nanmean(np.abs(np.asarray(v))[..., m, :] ** 2)))


def vlim_of(Z, R, r_lim, pct=98):
    m = (R >= r_lim[0]) & (R <= r_lim[1])
    return max(float(np.nanpercentile(np.abs(np.asarray(Z)[..., m, :]), pct)), 1e-6)


def page1_figure(part, dec, Zc, thetas, labels, X, Y, R, r_lim, zlabel):
    """Page-1 summary for one Zernike component: 5 per-rotator data maps +
    O (telescope) + C (camera) + residual, every panel annotated with its
    map RMS.  `part` is np.real (cos/scalar) or np.imag (sine partner).
    `model = data - residual` per rotator.
    """
    O = part(dec['O_pol']); C = part(dec['C_pol'])
    data = part(np.asarray(Zc)); resid = part(dec['res'])
    model = data - resid
    vlim = vlim_of(data, R, r_lim)
    mid = len(labels) // 2
    fig, axs = plt.subplots(2, 4, figsize=(20, 10.5), layout='constrained')
    for i in range(len(labels)):
        rd = map_rms(data[i], R, r_lim); rm = map_rms(model[i], R, r_lim)
        rr = map_rms(resid[i], R, r_lim)
        im = plot_field(axs.flat[i], X, Y, data[i],
                        f'data {zlabel} {labels[i]} (rot={thetas[i]:+.0f})\n'
                        f'RMS  data={rd:.4f}  model={rm:.4f}  res={rr:.4f}', vlim)
    plot_field(axs.flat[5], X, Y, O,
               f'O telescope (OCS)\n|O| RMS={map_rms(O, R, r_lim):.4f}', vlim)
    plot_field(axs.flat[6], X, Y, C,
               f'C camera (CCS)\n|C| RMS={map_rms(C, R, r_lim):.4f}', vlim)
    im = plot_field(axs.flat[7], X, Y, resid[mid],
                    f'residual {labels[mid]}\nRMS={map_rms(resid[mid], R, r_lim):.4f}',
                    vlim)
    fig.colorbar(im, ax=axs.ravel().tolist(), shrink=0.5, label=f'{zlabel} [um]')
    fig.suptitle(f'{zlabel}: telescope (OCS) + camera (CCS) split   '
                 f'(spin n={dec["n_spin"]}, s={dec["s"]:+d}, '
                 f'm0->{dec["degen_assignment"]})', fontsize=13)
    return fig

<a id='data'></a>
## 4. Data Access

Load the intrinsic grid map for each rotator dataset and read the actual mean
rotator angle of the visits each was built from.

In [4]:
fits_table = None
if rotator_angle_source == 'data':
    cols = ['visit', 'rotator_angle', 'alt', 'day_obs', 'seq_num']
    parts = [pd.read_parquet(p, columns=cols) for p in fits_parquet_paths
             if Path(p).exists()]
    fits_table = (pd.concat(parts, ignore_index=True).drop_duplicates('visit')
                  if parts else None)

dsets, thetas, labels = [], [], []
for ds in datasets:
    gp = Path(intrinsic_base) / ds['subdir'] / grid_filename
    if not gp.exists():
        raise FileNotFoundError(gp)
    df = pd.read_parquet(gp)
    zk = np.vstack(df['zk'].values)
    nolls = [int(j) for j in np.asarray(df['nollIndices'][0]).tolist()]
    rec = {'thx': df['thx_deg'].values, 'thy': df['thy_deg'].values,
           'z4opt': df['z4_optical_OCS'].values}
    for j in noll_list:
        rec[f'z{j}'] = zk[:, nolls.index(j)]
    dsets.append(rec)
    center = float(np.mean(ds['rot_range']))
    if fits_table is not None:
        mean_rot, n_v, lo, hi = isp.mean_rotator(fits_table, ds['rot_range'],
                                                 ds['alt_range'])
        if not np.isfinite(mean_rot):
            mean_rot, n_v, lo, hi = center, 0, np.nan, np.nan
    else:
        mean_rot, n_v, lo, hi = center, 0, np.nan, np.nan
    thetas.append(mean_rot); labels.append(ds['label'])
    print(f"{ds['label']:7s}: {len(df):4d} grid pts, rotator mean={mean_rot:+.2f} "
          f"deg (n={n_v}, dir-center {center:+.1f})")
thetas = np.array(thetas)
print(f'\nNoll terms: {noll_list}')
print(f'Rotator angles (deg): {np.round(thetas, 2)}')

rot-60 : 3856 grid pts, rotator mean=-59.90 deg (n=60, dir-center -60.0)
rot-15 : 3874 grid pts, rotator mean=-14.94 deg (n=53, dir-center -15.0)
rot00  : 3887 grid pts, rotator mean=-0.27 deg (n=217, dir-center +0.0)
rot+15 : 3898 grid pts, rotator mean=+14.43 deg (n=108, dir-center +15.0)
rot+60 : 3869 grid pts, rotator mean=+59.61 deg (n=93, dir-center +60.0)

Noll terms: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26]
Rotator angles (deg): [-59.9  -14.94  -0.27  14.43  59.61]


<a id='decomp'></a>
## 5. Decomposition

Sample every map onto a common polar grid, then solve `A_{d,m} = O_m +
C_m e^{-i m s theta_d}` per `(r, m)`.  The rotation sign is chosen
automatically (lower residual) unless fixed in the parameters.

### How the spin-aware Fourier fit works

Each rotator dataset *d* (rotator $\theta_d$) gives the intrinsic map in **OCS**
field coordinates.  A *telescope*-fixed aberration is constant in OCS; a
*camera*-fixed aberration rotates rigidly with the camera, in **both** the field
position **and** (for non-zero azimuthal order) the Zernike-coefficient
orientation.  For a Zernike of azimuthal order $n$ (the *spin*), forming the
complex field $A = Z_{\cos} + i\,Z_{\sin}$,

$$A_{\theta}(r,\phi) \;=\; O(r,\phi) \;+\; e^{\,i\,n\,s\theta}\,C\!\left(r,\;\phi - s\theta\right),$$

where $O$ is telescope-fixed (OCS), $C$ camera-fixed (CCS), and $s$ the
field-rotation sense ($s=+1$ here, geometry-confirmed).  The $e^{i n s\theta}$
factor is the Zernike-component rotation: a camera-fixed astig (spin 2) /
coma (spin 1) appears in OCS rotated by $2\theta$ / $1\theta$.  For a scalar
term ($n=0$: Z4, Z11, Z22) it is absent and the field is real.

**Diagonal in azimuthal Fourier.** Expanding in $e^{im\phi}$,

$$A_{d,m}(r) \;=\; O_m(r) \;+\; C_m(r)\,e^{\,i\,(n-m)\,s\theta_d},$$

a two-unknown least-squares solve per $(r,m)$ across the rotator datasets.

**Degeneracy at $m=n$.** When $m=n$ the phase is 1 for every dataset, so only
$O_n+C_n$ is determined — the generalisation of Z4's axisymmetric ($m=0$)
ambiguity.  `degen_assignment` puts it in $O$ (default), $C$, or splits it.

**Hole-aware fit.** `decompose_spin_lsq` fits $O$ and $C$ by least squares over
only the valid (non-hole) samples per ring (dead CCD/amp regions are
camera-fixed and rotate through OCS, so every OCS azimuth is sampled by some
dataset → $O$ is reconstructed hole-free; the camera-dead CCS regions are
filled by the smooth reconstruction of $C$).  `ridge` damps azimuthal modes the
ring coverage cannot constrain.

In [5]:
R, A, X, Y = isp.make_polar_grid(**polar)
th_rad = np.deg2rad(thetas)

def sample_field(key):
    """Polar-sample one component map across datasets, with hole masking."""
    maps = []
    for d in dsets:
        v = d[key]; f = np.isfinite(v)
        maps.append((d['thx'][f], d['thy'][f], v[f]))
    Z, valid, _ = isp.sample_maps_polar(maps, X, Y, hole_dist=hole_dist)
    return np.nan_to_num(Z), valid

# Field-rotation sense s (from Z4 unless fixed in params).
if rotation_sign == 'auto':
    _Z, _V = sample_field('z4opt' if use_z4_optical else 'z4')
    _r, _rms = isp.decompose_auto_sign(_Z, th_rad, A, R, r_lim=r_lim, valid=_V,
                                       method='lsq', m_max=m_max, ridge=ridge)
    s = _r['s']
    print(f's (field rotation) = {s:+d}   (Z4 auto: +1 {_rms[1]:.4f}, '
          f'-1 {_rms[-1]:.4f})')
else:
    s = int(rotation_sign); print(f's = {s:+d} (fixed)')

groups = isp.group_zernikes(noll_list)
dec_by_j = {}; metrics = []
for grp in groups:
    n_spin = spin_sign * grp['spin']
    if grp['kind'] == 'single':
        key = 'z4opt' if (grp['j'] == 4 and use_z4_optical) else f"z{grp['j']}"
        Z, V = sample_field(key)
        dec = isp.decompose_spin_lsq(Z.astype(complex), V, th_rad, A, n_spin=0,
                                     s=s, m_max=m_max, ridge=ridge,
                                     degen_assignment=degen_assignment)
        comps = [(grp['j'], np.real)]; Zc = Z.astype(complex)
    elif grp['kind'] == 'pair':
        Zc_, Vc_ = sample_field(f"z{grp['j_cos']}")
        Zs_, Vs_ = sample_field(f"z{grp['j_sin']}")
        Zc = Zc_ + 1j * Zs_; V = Vc_ & Vs_
        dec = isp.decompose_spin_lsq(Zc, V, th_rad, A, n_spin=n_spin, s=s,
                                     m_max=m_max, ridge=ridge,
                                     degen_assignment=degen_assignment)
        comps = [(grp['j_cos'], np.real), (grp['j_sin'], np.imag)]
    else:
        print(f"  (skipping unpaired {grp['label']})"); continue
    for j, part in comps:
        dec_by_j[j] = (dec, part, Zc)
        metrics.append(dict(
            Zernike=f'Z{j}', j=j, spin=abs(dec['n_spin']),
            dataRMS=map_rms(part(Zc), R, r_lim),
            O_tel=map_rms(part(dec['O_pol']), R, r_lim),
            C_cam=map_rms(part(dec['C_pol']), R, r_lim),
            residRMS=map_rms(part(dec['res']), R, r_lim)))

metrics_df = pd.DataFrame(metrics).sort_values('j').reset_index(drop=True)
with pd.option_context('display.float_format', '{:.4f}'.format):
    print('\n', metrics_df[['Zernike', 'spin', 'dataRMS', 'O_tel',
                             'C_cam', 'residRMS']].to_string(index=False))
print(f"\noverall: |O| telescope RMS={np.sqrt((metrics_df['O_tel']**2).mean()):.4f}  "
      f"|C| camera RMS={np.sqrt((metrics_df['C_cam']**2).mean()):.4f}  um")

s (field rotation) = +1   (Z4 auto: +1 0.0191, -1 0.0268)



 Zernike  spin  dataRMS  O_tel  C_cam  residRMS
     Z4     0   0.0462 0.0361 0.0219    0.0191
     Z5     2   0.0894 0.0818 0.0242    0.0296
     Z6     2   0.1009 0.0929 0.0287    0.0252
     Z7     1   0.1245 0.1125 0.0340    0.0350
     Z8     1   0.0855 0.0722 0.0412    0.0324
     Z9     3   0.0347 0.0308 0.0104    0.0127
    Z10     3   0.0401 0.0356 0.0091    0.0122
    Z11     0   0.0302 0.0269 0.0051    0.0131
    Z12     2   0.0238 0.0222 0.0068    0.0065
    Z13     2   0.0250 0.0227 0.0056    0.0085
    Z14     4   0.0172 0.0140 0.0060    0.0068
    Z15     4   0.0134 0.0112 0.0046    0.0061
    Z16     1   0.0247 0.0238 0.0046    0.0063
    Z17     1   0.0335 0.0319 0.0049    0.0068
    Z18     3   0.0128 0.0109 0.0043    0.0049
    Z19     3   0.0136 0.0121 0.0053    0.0050
    Z22     0   0.0228 0.0071 0.0031    0.0215
    Z23     2   0.0114 0.0109 0.0029    0.0032
    Z24     2   0.0166 0.0153 0.0031    0.0039
    Z25     4   0.0198 0.0195 0.0023    0.0038
    Z26    

<a id='plots'></a>
## 6. Maps & RMS summary

A telescope-vs-camera RMS summary across all Zernikes, then one page-1 panel
per Zernike (5 per-rotator data maps + O + C + residual) with the map RMS of
data / model / residual / |O| / |C| annotated on every panel.  Streamed to
`intrinsic_split_allz.pdf`.

In [6]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
n_pages = 0
with PdfPages(output_pdf) as pdf:
    # Summary: telescope (O) vs camera (C) map RMS per Zernike.
    d = metrics_df
    x = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(16, 5.5), layout='constrained')
    ax.bar(x - 0.21, d['O_tel'], 0.4, label='|O| telescope', color='steelblue')
    ax.bar(x + 0.21, d['C_cam'], 0.4, label='|C| camera', color='indianred')
    ax.plot(x, d['dataRMS'], 'k_', ms=12, label='data RMS')
    ax.plot(x, d['residRMS'], 'kx', ms=6, label='residual RMS')
    ax.set_xticks(x); ax.set_xticklabels(d['Zernike'], rotation=45)
    ax.set_ylabel('map RMS [um]')
    ax.set_title('Telescope (O) vs camera (C) map RMS per Zernike  '
                 '(in-window r=%.1f..%.1f deg)' % r_lim)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    pdf.savefig(fig, bbox_inches='tight'); plt.show(); plt.close(fig); n_pages += 1

    for j in noll_list:
        if j not in dec_by_j:
            continue
        dec, part, Zc = dec_by_j[j]
        fig = page1_figure(part, dec, Zc, thetas, labels, X, Y, R, r_lim, f'Z{j}')
        pdf.savefig(fig, bbox_inches='tight')
        if j in (5, 7):           # show a couple inline
            plt.show()
        plt.close(fig); n_pages += 1
print(f'Wrote {n_pages} pages to {output_pdf}')

/var/folders/7z/rlsxn7l17_lcfvvjqws8_0ywzz2g7p/T/ipykernel_80340/1878957355.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pdf.savefig(fig, bbox_inches='tight'); plt.show(); plt.close(fig); n_pages += 1


/var/folders/7z/rlsxn7l17_lcfvvjqws8_0ywzz2g7p/T/ipykernel_80340/1878957355.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Wrote 22 pages to output/intrinsic_split/intrinsic_split_allz.pdf


<a id='output'></a>
## 7. Output

Resample O (OCS) and C (CCS) onto the rot=0 dataset's focal-plane grid and
write a parquet.  `O_<col>` is the telescope map at each OCS field point;
`C_<col>` is the camera map at each CCS field point (same field positions).

In [7]:
# Resample every O (OCS) and C (CCS) map onto the rot~0 dataset's grid.
i0 = int(np.argmin(np.abs(thetas)))
thx0, thy0 = dsets[i0]['thx'], dsets[i0]['thy']
out = {'thx_deg': thx0, 'thy_deg': thy0}
for j in noll_list:
    if j not in dec_by_j:
        continue
    dec, part, _ = dec_by_j[j]
    out[f'O_Z{j}'] = isp.polar_field_to_points(part(dec['O_pol']), X, Y, thx0, thy0)
    out[f'C_Z{j}'] = isp.polar_field_to_points(part(dec['C_pol']), X, Y, thx0, thy0)
df_out = pd.DataFrame(out)
df_out.attrs['rotation_sign'] = int(s)
df_out.attrs['degen_assignment'] = degen_assignment
df_out.attrs['thetas_deg'] = list(np.round(thetas, 3))
Path(output_dir).mkdir(parents=True, exist_ok=True)
df_out.to_parquet(output_parquet)
print(f'Saved {output_parquet}: {len(df_out)} rows x {len(df_out.columns)} cols')
metrics_df.to_csv(f'{output_dir}/intrinsic_split_allz_rms.csv', index=False)
print(f'Saved RMS table: {output_dir}/intrinsic_split_allz_rms.csv')

Saved output/intrinsic_split/intrinsic_split_allz_maps.parquet: 3887 rows x 44 cols
Saved RMS table: output/intrinsic_split/intrinsic_split_allz_rms.csv
